In [ ]:
import os
import sys

# Move up one directory to the project root and add it to the Python path
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Now your imports will work perfectly!
from src.data_loader import load_insurance_data, calculate_insurance_metrics
from src.eda_utils import get_missing_summary, plot_loss_ratio_by_dimension

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats

def run_categorical_chi2_test(df: pd.DataFrame, feature_col: str, group_a: str, group_b: str) -> tuple:
    """Performs chi-square independence evaluation matching claim occurrences across distinct category assignments."""
    # Filter for target groups
    sub_df = df[df[feature_col].isin([group_a, group_b])].copy()
    sub_df['HasClaim'] = np.where(sub_df['TotalClaims'] > 0, 1, 0)
    
    contingency_table = pd.crosstab(sub_df[feature_col], sub_df['HasClaim'])
    chi2, p_val, dof, expected = stats.chi2_contingency(contingency_table)
    return chi2, p_val

def run_numerical_ttest(df: pd.DataFrame, feature_col: str, group_a: str, group_b: str, metric_col: str) -> tuple:
    """Executes a two-sample t-test evaluating expected variance of numeric indicators between groups."""
    data_a = df[df[feature_col] == group_a][metric_col].dropna()
    data_b = df[df[feature_col] == group_b][metric_col].dropna()
    
    # Using Welch's t-test (equal_var=False) given structural differences in size and shape
    t_stat, p_val = stats.ttest_ind(data_a, data_b, equal_var=False)
    return t_stat, p_val


In [ ]:
# 2. Analytical Implementation Framework 
# Execution code within notebook context
import pandas as pd
from src.data_loader import load_insurance_data, calculate_insurance_metrics
from src.hypothesis_tests import run_categorical_chi2_test, run_numerical_ttest

df = load_insurance_data('../data/insurance_data.csv')
df = calculate_insurance_metrics(df)

results_summary = []

# Example Test 1: Gender vs Claim Frequency
chi2_g, p_g = run_categorical_chi2_test(df, 'Gender', 'Male', 'Female')
results_summary.append({
    'Hypothesis': 'Risk variance across Gender labels',
    'Test Type': 'Chi-Square Contingency',
    'p-value': p_g,
    'Decision': 'Reject H0' if p_g < 0.05 else 'Fail to Reject H0'
})

# Example Test 2: Selected Province Pairs on Claim Severity (Only considering positive claims records)
claims_only = df[df['TotalClaims'] > 0]
if len(df['Province'].unique()) >= 2:
    p1, p2 = df['Province'].value_counts().index[0], df['Province'].value_counts().index[1]
    t_p, p_p = run_numerical_ttest(claims_only, 'Province', p1, p2, 'TotalClaims')
    results_summary.append({
        'Hypothesis': f'Claim Severity Variance ({p1} vs {p2})',
        'Test Type': 'Welch t-test',
        'p-value': p_p,
        'Decision': 'Reject H0' if p_p < 0.05 else 'Fail to Reject H0'
})

print(pd.DataFrame(results_summary).to_markdown(index=False))
